In [130]:
#import libraries
import pandas as pd
import sqlite3

In [131]:
#load the processed sales data into a dataframe
df = pd.read_csv("../data/cleaned/processed_sales_data.csv")
df.head()

,Unnamed: 0,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,...,discount,profit,order_year,order_month,order_month_num,order_day_of_week,weekend_flag,quarter,profit_margin,sales_band
0,0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,...,0.00,41.9136,2016,November,11,Tuesday,False,4,16.00,High
1,1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,...,0.00,219.5820,2016,November,11,Tuesday,False,4,30.00,Very High
2,2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,...,0.00,6.8714,2016,June,6,Sunday,True,2,47.00,Very Low
3,3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,...,0.45,-383.0310,2015,October,10,Sunday,True,4,-40.00,Premium
4,4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,...,0.20,2.5164,2015,October,10,Sunday,True,4,11.25,Low


In [132]:
df.columns

Index(['Unnamed: 0', 'row_id', 'order_id', 'order_date', 'ship_date',
       'ship_mode', 'customer_id', 'customer_name', 'segment', 'country',
       'city', 'state', 'postal_code', 'region', 'product_id', 'category',
       'sub-category', 'product_name', 'sales', 'quantity', 'discount',
       'profit', 'order_year', 'order_month', 'order_month_num',
       'order_day_of_week', 'weekend_flag', 'quarter', 'profit_margin',
       'sales_band'],
      dtype='object')

In [133]:
#create a sqlite database connection
conn = sqlite3.connect("../data/cleaned/sales_analysis.db")
cursor = conn.cursor()

In [134]:
#drop unnecessary index column
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
print(df)

      row_id        order_id  order_date   ship_date       ship_mode  \
0          1  CA-2016-152156  2016-11-08  2016-11-11    Second Class   
1          2  CA-2016-152156  2016-11-08  2016-11-11    Second Class   
2          3  CA-2016-138688  2016-06-12  2016-06-16    Second Class   
3          4  US-2015-108966  2015-10-11  2015-10-18  Standard Class   
4          5  US-2015-108966  2015-10-11  2015-10-18  Standard Class   
...      ...             ...         ...         ...             ...   
9989    9990  CA-2014-110422  2014-01-21  2014-01-23    Second Class   
9990    9991  CA-2017-121258  2017-02-26  2017-03-03  Standard Class   
9991    9992  CA-2017-121258  2017-02-26  2017-03-03  Standard Class   
9992    9993  CA-2017-121258  2017-02-26  2017-03-03  Standard Class   
9993    9994  CA-2017-119914  2017-05-04  2017-05-09    Second Class   

     customer_id     customer_name    segment        country             city  \
0       CG-12520       Claire Gute   Consumer  United 

In [135]:
#convert order_date to datetime format
df["order_date"] = pd.to_datetime(df["order_date"])
df["order_date"]

0      2016-11-08
1      2016-11-08
2      2016-06-12
3      2015-10-11
4      2015-10-11
          ...    
9989   2014-01-21
9990   2017-02-26
9991   2017-02-26
9992   2017-02-26
9993   2017-05-04
Name: order_date, Length: 9994, dtype: datetime64[ns]

In [136]:
#load the dataframe into a sql table
df.to_sql("sales", conn, if_exists="replace", index=False)

9994

In [137]:
#we run a query to approve the data was loaded
pd.read_sql("SELECT * FROM sales LIMIT 5;", conn)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,discount,profit,order_year,order_month,order_month_num,order_day_of_week,weekend_flag,quarter,profit_margin,sales_band
0,1,CA-2016-152156,2016-11-08 00:00:00,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,41.9136,2016,November,11,Tuesday,0,4,16.00,High
1,2,CA-2016-152156,2016-11-08 00:00:00,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,219.5820,2016,November,11,Tuesday,0,4,30.00,Very High
2,3,CA-2016-138688,2016-06-12 00:00:00,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,0.00,6.8714,2016,June,6,Sunday,1,2,47.00,Very Low
3,4,US-2015-108966,2015-10-11 00:00:00,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.45,-383.0310,2015,October,10,Sunday,1,4,-40.00,Premium
4,5,US-2015-108966,2015-10-11 00:00:00,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.20,2.5164,2015,October,10,Sunday,1,4,11.25,Low


### Business question

- Which categories generated the highest sales and profit

In [138]:
#aggregate total sales and profits by category
df = pd.read_sql("""
SELECT Category,
       SUM(Sales) AS total_sales,
       SUM(Profit) AS total_profit
FROM sales
GROUP BY Category
ORDER BY total_sales DESC;          
""", conn)
df

,category,total_sales,total_profit
0,Technology,836154.0330,145454.9481
1,Furniture,741999.7953,18451.2728
2,Office Supplies,719047.0320,122490.8008


In [139]:
#business question 2 we calculate sales and profits by region
df = pd.read_sql("""
SELECT Region,
       SUM(Sales) AS total_sales,
       SUM(Profit) AS total_profit
FROM sales
GROUP BY Region
ORDER BY total_profit DESC;                 
""", conn)

df

,region,total_sales,total_profit
0,West,725457.8245,108418.4489
1,East,678781.2400,91522.7800
2,South,391721.9050,46749.4303
3,Central,501239.8908,39706.3625


In [140]:
#business question3  we identify the top 10 most profitable products
df = pd.read_sql("""
SELECT Product_Name,
       ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY Product_Name
ORDER BY total_profit DESC
LIMIT 10;                
""", conn)
df

,product_name,total_profit
0,Canon imageCLASS 2200 Advanced Copier,25199.93
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,7753.04
2,Hewlett Packard LaserJet 3310 Copier,6983.88
3,Canon PC1060 Personal Laser Copier,4570.93
4,HP Designjet T520 Inkjet Large Format Printer ...,4094.98
5,Ativa V4110MDD Micro-Cut Shredder,3772.95
6,"3D Systems Cube Printer, 2nd Generation, Magenta",3717.97
7,Plantronics Savi W720 Multi-Device Wireless He...,3696.28
8,Ibico EPK-21 Electric Binding System,3345.28
9,Zebra ZM400 Thermal Label Printer,3343.54


In [141]:
#we analyze sales trends over time using order date
df = pd.read_sql("""
SELECT Order_year,
       ROUND(SUM(Sales), 2) AS total_sales
FROM sales
GROUP BY Order_year
ORDER BY Order_year;                
""", conn)

df

,order_year,total_sales
0,2014,484247.50
1,2015,470532.51
2,2016,609205.60
3,2017,733215.26


In [142]:
#we analyze how different discount levels if it will affect profit
df = pd.read_sql("""
SELECT Discount,
       AVG(Profit) AS avg_profit
FROM sales
GROUP BY Discount
ORDER BY Discount;                 
""", conn)

df

,discount,avg_profit
0,0.00,66.900292
1,0.10,96.055074
2,0.15,27.288298
3,0.20,24.702572
4,0.30,-45.679636
5,0.32,-88.560656
6,0.40,-111.927429
7,0.45,-226.646464
8,0.50,-310.703456
9,0.60,-43.077212


In [143]:
#we analyze which customer segments generate the most revenue
df = pd.read_sql("""
SELECT Segment,
       SUM(Sales) AS total_sales
FROM sales
GROUP BY Segment
ORDER BY total_sales DESC;              
""", conn)

df

,segment,total_sales
0,Consumer,1.161401e+06
1,Corporate,7.061464e+05
2,Home Office,4.296531e+05


In [144]:
#save my queries into queries.sql
queries = """
SELECT Category,
       SUM(Sales) AS total_sales,
       SUM(Profit) AS total_profit
FROM sales
GROUP BY Category
ORDER BY total_sales DESC;          

SELECT Region,
       SUM(Sales) AS total_sales,
       SUM(Profit) AS total_profit
FROM sales
GROUP BY Region
ORDER BY total_sales DESC;

SELECT Product_Name,
       ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY Product_Name
ORDER BY total_profit DESC
LIMIT 10;

SELECT Order_year,
       SUM(Sales) AS total_sales
FROM sales
GROUP BY Order_year
ORDER BY Order_year;

SELECT Discount,
       AVG(Profit) AS avg_profit
FROM sales
GROUP BY Discount
ORDER BY Discount;

SELECT Segment,
       SUM(Sales) AS total_sales
FROM sales
GROUP BY Segment
ORDER BY total_sales DESC;
   
"""
with open("../data/cleaned/queries.sql", "w") as file:
    file.write(queries)

In [145]:
conn.close()